# 03. Export to Claude Desktop

`papers_scored.csv` 상위 N개를 markdown 표로 export → Claude Desktop 채팅에 붙여넣어 `reference_quality_check(rr)` 또는 `literature_synthesis(rr)` 프롬프트로 사용.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_scored.csv')

TOP_N = 15
top = df.head(TOP_N).copy()
top.head()

In [ ]:
def to_markdown(df):
    lines = ['| # | 제목 | 저자 | 연도 | 인용수 | 점수 | DOI |',
             '|---|---|---|---|---|---|---|']
    for i, row in df.reset_index(drop=True).iterrows():
        title = (row.get('title') or '').replace('|', '\\|')[:120]
        authors = (row.get('authors') or '').replace('|', '\\|')[:80]
        year = row.get('year', '')
        cit = int(row.get('cited_by_count', 0) or 0)
        score = row.get('quality_score', '')
        doi = row.get('doi', '') or ''
        lines.append(f'| {i+1} | {title} | {authors} | {year} | {cit} | {score} | {doi} |')

    abstracts = ['', '## Abstracts', '']
    for i, row in df.reset_index(drop=True).iterrows():
        abs_text = (row.get('abstract') or '(no abstract)')
        abstracts.append(f'### [{i+1}] {row.get("title")}')
        abstracts.append(abs_text)
        abstracts.append('')
    return '\n'.join(lines + abstracts)

md = to_markdown(top)
out = DATA_DIR / f'papers_top_{TOP_N}.md'
out.write_text(md, encoding='utf-8')
print(f'Saved → {out.resolve()}')
print('\n--- Preview ---\n')
print(md[:1200])

## 다음 단계

1. `data/papers_top_15.md` 파일을 열어 전체 내용을 복사.
2. Claude Desktop에서 `reference_quality_check(rr)` 프롬프트와 함께 붙여넣기.
3. 🟢 분류된 논문은 본문을 직접 PDF로 다운받아 읽기 (DOI 또는 OA URL 사용).
4. 본문에서 발견한 핵심을 한 줄씩 정리한 뒤 `literature_synthesis(rr)` 호출.